In this lab, the objective is to identify the customers who were active in both May and June, and how did their activity differ between months. To achieve this, follow these steps:

1. Establish a connection between Python and the Sakila database.

In [1]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine
import mysql.connector

from getpass import getpass

In [2]:
pwd = getpass()

In [10]:
connection = mysql.connector.connect(host = "localhost",
                                     user = "root",
                                     passwd = pwd)


In [6]:
user = "root"
host = "localhost"

engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}")

In [11]:
connection.is_connected()

True

2. Write a Python function called `rentals_month` that retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame. The function should take in three parameters:

	- `engine`: an object representing the database connection engine to be used to establish a connection to the Sakila database.
	- `month`: an integer representing the month for which rental data is to be retrieved.
	- `year`: an integer representing the year for which rental data is to be retrieved.

	The function should execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, and return it as a pandas DataFrame.



In [ ]:
def select_schema():
    schema_name = input('Please enter the name of the schema you want to analyze')
    cursor = connection.cursor()
    cursor.execute(f"""
                    Use {schema_name}
                    """)

In [42]:
def rentals_month(month_int, year_int):
    cursor = connection.cursor()
    cursor.execute(f"""
                    SELECT MONTH(rental_date) AS month, YEAR(rental_date) AS year 
                    FROM rental
                    WHERE MONTH(rental_date) = {month_int} AND YEAR(rental_date) = {year_int};
                    """)
    result = cursor.fetchall()
    return pd.DataFrame(result, columns=cursor.column_names)

3. Develop a Python function called `rental_count_month` that takes the DataFrame provided by `rentals_month` as input along with the month and year and returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year. 

	The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".


	*Hint: Consider making use of pandas [groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)*

In [59]:
def rental_count_month(month_int, year_int):
    df = rentals_month(month_int, year_int)
    col_name = f'rentals_{month_int}_{year_int}'
    df = df.groupby(['month', 'year']).agg(month_count=('month', 'count')).reset_index()
    df.rename(columns={'month_count': col_name}, inplace=True)
    return df

4. Create a Python function called `compare_rentals` that takes two DataFrames as input containing the number of rentals made by each customer in different months and years. 
The function should return a combined DataFrame with a new 'difference' column, which is the difference between the number of rentals in the two months.

In [ ]:
def compare_rentals(month_int, year_int):
    cursor = connection.cursor()
    cursor.execute(f"""
                    WITH 
                    current_m AS (SELECT customer_id, COUNT(*) AS current_rentals
                        FROM rental
                        WHERE MONTH(rental_date) = {month_int} AND YEAR(rental_date) = {year_int}
                        GROUP BY customer_id),
                    previous_m AS (SELECT customer_id, COUNT(*) AS previous_rentals
                        FROM rental
                        WHERE MONTH(rental_date) = {month_int-1} AND YEAR(rental_date) = {year_int}
                        GROUP BY customer_id)

                    SELECT 
                        COALESCE(c.customer_id, p.customer_id) AS customer_id,
                        COALESCE(p.previous_rentals,0) AS month_{month_int-1}_rentals,
                        COALESCE(c.current_rentals,0) AS month_{month_int}_rentals
                    FROM current_m AS c
                    LEFT JOIN previous_m AS p 
                        ON c.customer_id = p.customer_id

                    UNION

                    SELECT 
                        COALESCE(c.customer_id, p.customer_id) AS customer_id,
                        COALESCE(p.previous_rentals,0) AS month_{month_int-1}_rentals,
                        COALESCE(c.current_rentals,0) AS month_{month_int}_rentals
                    FROM previous_m AS p
                    LEFT JOIN current_m AS c 
                        ON p.customer_id = c.customer_id;
                    """)
    result = cursor.fetchall()
    df = pd.DataFrame(result, columns=cursor.column_names)
    df['diff_rentals'] = df[df.columns[-1]] - df[df.columns[-2]]
    return df

In [75]:
def get_user_input():
    month = input('Enter the month number you want to analyze:')
    year = input('Enter the year you want to analyze:')

    display(rental_count_month(month, year))
    display(compare_rentals(month, year))